# Switched-Array Radio Direction-Finding System

This notebook implements the complete hardware radio direction-finding system.

The system uses a permanent reference antenna connected to Pluto RX1 and a
set of antennas that are sequentially switched into Pluto RX2. An STM32
controls the RF switch, while Julia coordinates the switch states, acquires
the two receive channels, processes the I/Q data and estimates the direction
of the received signal.

The implementation is kept configurable so that the number of switched
antennas, operating frequency and array geometry can be changed without
restructuring the processing code.

In [ ]:
# Python environment used by pyadi-iio

pluto_python = raw"C:\Users\thoba\Documents\PlutoPythonTest\.venv\Scripts\python.exe"

ENV["JULIA_CONDAPKG_BACKEND"] = "Null"
ENV["JULIA_PYTHONCALL_EXE"] = pluto_python

In [ ]:
# Packages

using PythonCall
using Statistics
using FFTW
using LinearAlgebra
using Plots
using LibSerialPort
plotly()

adi = pyimport("adi")
subprocess = pyimport("subprocess")

In [ ]:
# ============================================================
# Radio settings
# ============================================================

sample_rate = 1_000_000           # samples
centre_frequency = 91_900_000     # Hz
rf_bandwidth = 800_000            # Hz
rx_buffer_size = 4096
rx_gain = 30.0                    # dB


# ============================================================
# Signal settings
# ============================================================

signal_frequency = 92_100_000     # Hz
signal_bandwidth = 200_000        # Hz

# ============================================================
# Array settings
# ============================================================

c = 3e8
wavelength = c / signal_frequency

antenna_spacing = 0.4             # metres

reference_antenna = "A1"

# HMC7992 switch path order
# path 1 [A2]
# path 2 [A3]
# path 3 [A4]
# path 4 [A5]

switched_antennas = ["A2", "A3", "A4", "A5"]

num_switched_antennas = length(switched_antennas)

# ============================================================
# STM32 settings
# ============================================================

stm_port = "COM8"
stm_baud = 115200

discard_buffers = 5

## Hardware Connections

In [ ]:
# Connect to the STM32 over the USB serial port

stm = LibSerialPort.open(stm_port, stm_baud)

println("STM32 connected on ", stm_port)

In [ ]:
# Find the Pluto Ethernet address

result = subprocess.run(
    ["iio_info", "-s"],
    capture_output = true,
    text = true
)

output = pyconvert(String, result.stdout)

pluto_ip = nothing

for line in split(output, '\n')

    if occursin("Pluto", line)

        for match in eachmatch(r"\b(?:\d{1,3}\.){3}\d{1,3}\b", line)

            ip = match.match

            # Ignore the Pluto USB-network address
            if ip != "192.168.2.1"
                pluto_ip = ip
                break
            end
        end
    end
end

pluto_ip === nothing &&
    error("Pluto Ethernet connection not found.")

pluto_uri = "ip:" * pluto_ip

sdr = adi.ad9361(uri = pluto_uri)

println("Pluto connected at ", pluto_ip)

In [ ]:
function configure_rdf_receiver(sdr)

    # Use both receive channels
    pysetattr(sdr, "rx_enabled_channels", pylist([0, 1]))

    # RDF receiver settings
    pysetattr(sdr, "sample_rate", sample_rate)
    pysetattr(sdr, "rx_lo", centre_frequency)
    pysetattr(sdr, "rx_rf_bandwidth", rf_bandwidth)
    pysetattr(sdr, "rx_buffer_size", rx_buffer_size)

    # Manual gain on both channels
    pysetattr(sdr, "gain_control_mode_chan0", "manual")
    pysetattr(sdr, "gain_control_mode_chan1", "manual")

    pysetattr(sdr, "rx_hardwaregain_chan0", rx_gain)
    pysetattr(sdr, "rx_hardwaregain_chan1", rx_gain)

    println("Pluto configured for RDF measurements")

    return nothing
end

In [ ]:
configure_rdf_receiver(sdr)

## RF Switch Control

In [ ]:
# Select one of the switched antenna paths

function select_path(stm, path)
    
    # Check that the requested path exists
    if path < 1 ||  path > num_switched_antennas
        error("Invalid RF path: $path")
    end
    
    # Send the path number to the STM32
    command = string(path)
    write(stm, command)
    
    # Wait until the STM32 confirms the switched state
    response = strip(readline(stm))
    expected_response =  "READY $path"
    
    if response != expected_response
        error("Expected '$expected_response', received '$response'")
    end
    
    return nothing
end

## RDF Measurement Workflow

In [ ]:
# Measure RX2 relative to the fixed reference on RX1

function measure_relative_phase(sdr)
    
    # Capture both Pluto receive channels simultaneously
    rx_data = sdr.rx()
    
    rx1 = pyconvert(Vector{ComplexF64}, rx_data[0])
    rx2 = pyconvert(Vector{ComplexF64}, rx_data[1])
    
    # Remove DC offset
    rx1 = rx1 .- mean(rx1)
    rx2 = rx2 .- mean(rx2)
    
    # Relative complex phasor between RX2 and RX1
    cross_signal = sum(rx2 .* conj.(rx1))
    
    energy_rx1 = sum(abs2, rx1)
    energy_rx2 = sum(abs2, rx2)
    
    relative_phasor = cross_signal / sqrt(energy_rx1 * energy_rx2)
    
    phase_deg = rad2deg(angle(relative_phasor))
    correlation = abs(relative_phasor)
    gain_db = 10 * log10(energy_rx2 / energy_rx1)
    
    return (
        phase_deg = phase_deg,
        correlation = correlation,
        gain_db = gain_db  
    )
    
end

## Optional Utility: VCO Source Characterisation

The VCO carrier frequency is not known exactly because it depends on the
tuning voltage. The Pluto is therefore swept across a wide frequency range
and the received spectrum is examined to locate the strongest RF carrier.

In [ ]:
# Frequency range to search for the VCO carrier

sweep_start = 650_000_000       # 650 MHz
sweep_stop  = 1_100_000_000     # 1100 MHz

sweep_sample_rate = 5_000_000   # 5 MS/s
sweep_bandwidth = 4_000_000     # 4 MHz
sweep_step = 4_000_000          # Move 4 MHz at a time

sweep_buffer_size = 4096

# Centre frequencies that the Pluto will visit during the sweep

sweep_frequencies = sweep_start : sweep_step : sweep_stop

In [ ]:
# Configure Pluto for the VCO frequency sweep

sweep_gain = 10.0     

pysetattr(sdr, "rx_enabled_channels", pylist([0]))
pysetattr(sdr, "sample_rate", sweep_sample_rate)
pysetattr(sdr, "rx_rf_bandwidth", sweep_bandwidth)
pysetattr(sdr, "rx_buffer_size", sweep_buffer_size)

pysetattr(sdr, "gain_control_mode_chan0", "manual")
pysetattr(sdr, "rx_hardwaregain_chan0", sweep_gain)

println("Sweep sample rate: ", sweep_sample_rate / 1e6, " MS/s")
println("Sweep bandwidth:   ", sweep_bandwidth / 1e6, " MHz")
println("Sweep RX gain:     ", sweep_gain, " dB")

In [ ]:
# Storage for the complete measured spectrum

all_frequencies = Float64[]
all_magnitudes = Float64[]


for centre_freq in sweep_frequencies
    
    # Tune Pluto to the next centre frequency
    pysetattr(sdr, "rx_lo", centre_freq)
    
    # Discard one capture after changing frequency
    # so that the next capture belongs to the new tuning position
    sdr.rx()
    
    # Capture IQ samples
    rx_data = sdr.rx()
    rx = pyconvert(Vector{ComplexF64}, rx_data)
    
    # Remove DC offset
    rx = rx .- mean(rx)
    
    # Calculate the spectrum
    N = length(rx)
    
    frequency_bins = -N÷2 : N÷2-1
    baseband_frequency = frequency_bins .* (sweep_sample_rate / N)
    
    spectrum = abs.(fftshift(fft(rx)))
    
    # Convert baseband frequency to actual RF frequency
    rf_frequency = centre_freq .+ baseband_frequency

    # Keep only the central 4 MHz of this capture
    keep = abs.(baseband_frequency) .<= sweep_bandwidth / 2

    append!(all_frequencies, rf_frequency[keep])
    append!(all_magnitudes, spectrum[keep])
end

println("Sweep complete")

In [ ]:
peak_index = argmax(all_magnitudes)

peak_frequency = all_frequencies[peak_index]
peak_magnitude = all_magnitudes[peak_index]

println("Strongest detected frequency: ", round(peak_frequency / 1e6, digits = 3), " MHz")

In [ ]:
# Plot the FFT magnitude across the complete sweep

plot(
    all_frequencies ./ 1e6,
    all_magnitudes,

    xlabel = "Frequency (MHz)",
    ylabel = "FFT Magnitude",
    title = "VCO Frequency Spectrum",

    legend = false,
    linewidth = 1
)

In [ ]:
# Restore the Pluto to the normal RDF receiver configuration
configure_rdf_receiver(sdr)